In [1]:
import os
import re
import sys
import json
import numpy as np
import pandas as pd
import ray
from tqdm import tqdm

# -------------------------
# TensorRT version normalize
# -------------------------
try:
    import tensorrt as trt
    m = re.match(r"^\d+(?:\.\d+)*", trt.__version__)
    trt.__version__ = m.group(0) if m else trt.__version__
except Exception:
    pass

# -------------------------
# Paths / Config
# -------------------------
INPUT_DIR = "./data/2_frame_files"
OUTPUT_DIR = "./data/3_ocr_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CRAFT_REPO_DIR = "./CRAFT-pytorch"
CRAFT_WEIGHTS_PATH = "./CRAFT-pytorch/craft_mlt_25k.pth"

NUM_WORKERS = 8


# -------------------------
# Shared utilities
# -------------------------
def order_quad_clockwise(pts: np.ndarray) -> np.ndarray:
    pts = np.asarray(pts, dtype=np.float32)
    s = pts.sum(axis=1)
    d = np.diff(pts, axis=1).reshape(-1)
    tl = pts[np.argmin(s)]
    br = pts[np.argmax(s)]
    tr = pts[np.argmin(d)]
    bl = pts[np.argmax(d)]
    return np.array([tl, tr, br, bl], dtype=np.float32)


def polygon_to_xywha(cv2_mod, polygon: np.ndarray) -> list:
    rect = cv2_mod.minAreaRect(polygon.astype(np.float32))
    (cx, cy), (w, h), angle = rect
    if w < h:
        w, h = h, w
        angle = angle + 90
    if angle >= 90:
        angle -= 180
    if angle < -90:
        angle += 180
    return [float(cx), float(cy), float(w), float(h), float(angle)]


# -------------------------
# Ray Actor
# -------------------------
ray.init(num_gpus=1)


@ray.remote(num_gpus=1 / NUM_WORKERS)
class OCRWorker:
    def __init__(self):
        import cv2
        self.cv2 = cv2

        from paddleocr import PaddleOCR
        self.ocr = PaddleOCR(
            use_doc_orientation_classify=False,
            use_doc_unwarping=False,
            use_textline_orientation=False,
            text_detection_model_name="PP-OCRv5_mobile_det",
            text_recognition_model_name="korean_PP-OCRv5_mobile_rec",
            device="gpu:0",
            lang="korean",
            precision="fp16",
            #use_tensorrt=True,
            #enable_hpi=True,

        )

        # CRAFT lazy init
        self.torch = None
        self.net = None
        self.device = None
        self.canvas_size = 720
        self.mag_ratio = 1.5
        self.text_threshold = 0.4
        self.low_text = 0.25
        self.min_cc_area = 12
        self.pad_px = 4

    def _ensure_craft(self):
        if self.net is not None:
            return
        import torch
        self.torch = torch
        if CRAFT_REPO_DIR not in sys.path:
            sys.path.append(CRAFT_REPO_DIR)
        from craft import CRAFT
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.net = CRAFT()
        state = torch.load(CRAFT_WEIGHTS_PATH, map_location="cpu")
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        cleaned = {(k[7:] if k.startswith("module.") else k): v for k, v in state.items()}
        self.net.load_state_dict(cleaned, strict=True)
        self.net.eval()
        self.net.to(self.device)
        self.net.half()

    # ---------- PaddleOCR ----------
    def _normalize_page_result(self, page_result):
        if hasattr(page_result, "json"):
            candidate = page_result.json
            page_result = candidate() if callable(candidate) else candidate
        if isinstance(page_result, str):
            try:
                page_result = json.loads(page_result)
            except Exception:
                pass
        if isinstance(page_result, dict) and "res" in page_result:
            page_result = page_result["res"]
        if not isinstance(page_result, dict):
            raise TypeError(f"Unexpected result type: {type(page_result)}")
        return page_result

    # ---------- CRAFT ----------
    def _resize_aspect_ratio(self, img):
        h, w = img.shape[:2]
        target_size = self.mag_ratio * max(h, w)
        if target_size > self.canvas_size:
            target_size = self.canvas_size
        ratio = target_size / max(h, w)
        th, tw = max(1, int(h * ratio)), max(1, int(w * ratio))
        resized = self.cv2.resize(img, (tw, th), interpolation=self.cv2.INTER_LINEAR)
        th32 = th if th % 32 == 0 else th + (32 - th % 32)
        tw32 = tw if tw % 32 == 0 else tw + (32 - tw % 32)
        canvas = np.zeros((th32, tw32, 3), dtype=np.uint8)
        canvas[:th, :tw, :] = resized
        return canvas, ratio

    def _prepare_single(self, crop_bgr):
        crop_rgb = self.cv2.cvtColor(crop_bgr, self.cv2.COLOR_BGR2RGB)
        resized, ratio = self._resize_aspect_ratio(crop_rgb)
        img = resized.astype(np.float32)
        img -= np.array([0.485 * 255, 0.456 * 255, 0.406 * 255], dtype=np.float32)
        img /= np.array([0.229 * 255, 0.224 * 255, 0.225 * 255], dtype=np.float32)
        return img.transpose(2, 0, 1), ratio

    def _infer_textmaps_batch(self, crops_bgr):
        if not crops_bgr:
            return []
        self._ensure_craft()
        prepared, ratios = [], []
        for crop in crops_bgr:
            chw, ratio = self._prepare_single(crop)
            prepared.append(chw)
            ratios.append(ratio)
        max_h = max(p.shape[1] for p in prepared)
        max_w = max(p.shape[2] for p in prepared)
        batch = np.zeros((len(prepared), 3, max_h, max_w), dtype=np.float32)
        for i, p in enumerate(prepared):
            batch[i, :, :p.shape[1], :p.shape[2]] = p
        x = self.torch.from_numpy(batch).half().to(self.device)
        with self.torch.no_grad():
            y, _ = self.net(x)
        del x
        result = [(y[i, :, :, 0].float().cpu().numpy(), ratios[i]) for i in range(len(prepared))]
        del y
        self.torch.cuda.empty_cache()
        return result

    def _warp_crop(self, image_bgr, polygon):
        poly = order_quad_clockwise(polygon)
        w_top = np.linalg.norm(poly[1] - poly[0])
        w_bot = np.linalg.norm(poly[2] - poly[3])
        h_right = np.linalg.norm(poly[2] - poly[1])
        h_left = np.linalg.norm(poly[3] - poly[0])
        dst_w = max(8, int(round(max(w_top, w_bot))))
        dst_h = max(8, int(round(max(h_left, h_right))))
        dst = np.array([[0, 0], [dst_w-1, 0], [dst_w-1, dst_h-1], [0, dst_h-1]], dtype=np.float32)
        M = self.cv2.getPerspectiveTransform(poly, dst)
        Minv = self.cv2.getPerspectiveTransform(dst, poly)
        crop = self.cv2.warpPerspective(image_bgr, M, (dst_w, dst_h),
                                         flags=self.cv2.INTER_LINEAR, borderMode=self.cv2.BORDER_REPLICATE)
        if self.pad_px > 0:
            crop = self.cv2.copyMakeBorder(crop, self.pad_px, self.pad_px, self.pad_px, self.pad_px,
                                            borderType=self.cv2.BORDER_REPLICATE)
        if crop.shape[0] < 8 or crop.shape[1] < 8:
            return None
        return crop, dst_w, dst_h, Minv

    def _expand_polygon(self, poly, margin=0.05):
        center = poly.mean(axis=0)
        return (center + (poly - center) * (1.0 + 2.0 * margin)).astype(np.float32)

    def _extract_char_boxes(self, score_text, ratio, crop_w, crop_h):
        binary = (score_text > self.text_threshold).astype(np.uint8) * 255
        num_labels, labels, stats, _ = self.cv2.connectedComponentsWithStats(binary, connectivity=8)
        boxes = []
        m2c = 2.0 / ratio
        for k in range(1, num_labels):
            if stats[k, self.cv2.CC_STAT_AREA] < self.min_cc_area:
                continue
            sx, sy = stats[k, self.cv2.CC_STAT_LEFT], stats[k, self.cv2.CC_STAT_TOP]
            sw, sh = stats[k, self.cv2.CC_STAT_WIDTH], stats[k, self.cv2.CC_STAT_HEIGHT]
            if float(score_text[labels == k].mean()) < self.low_text:
                continue
            x0, y0 = max(0.0, sx * m2c), max(0.0, sy * m2c)
            x1 = min(float(crop_w - 1), (sx + sw) * m2c)
            y1 = min(float(crop_h - 1), (sy + sh) * m2c)
            if (x1 - x0) < 2 or (y1 - y0) < 2:
                continue
            boxes.append(np.array([[x0, y0], [x1, y0], [x1, y1], [x0, y1]], dtype=np.float32))
        boxes.sort(key=lambda b: (float(b[:, 0].mean()), float(b[:, 1].mean())))
        return boxes

    def _segment_polygons_batch(self, image_bgr, polygons):
        if not polygons:
            return []
        warp_results, valid_indices = [], []
        for i, poly in enumerate(polygons):
            r = self._warp_crop(image_bgr, poly)
            warp_results.append(r)
            if r is not None:
                valid_indices.append(i)
        if not valid_indices:
            return [[] for _ in polygons]
        valid_crops = [warp_results[i][0] for i in valid_indices]
        textmap_results = self._infer_textmaps_batch(valid_crops)
        all_items = [[] for _ in polygons]
        for bi, oi in enumerate(valid_indices):
            crop, dst_w, dst_h, Minv = warp_results[oi]
            score_text, ratio = textmap_results[bi]
            char_boxes = self._extract_char_boxes(score_text, ratio, crop.shape[1], crop.shape[0])
            items = []
            for box in char_boxes:
                bu = box.copy()
                if self.pad_px > 0:
                    bu[:, 0] -= self.pad_px
                    bu[:, 1] -= self.pad_px
                bu[:, 0] = np.clip(bu[:, 0], 0, dst_w - 1)
                bu[:, 1] = np.clip(bu[:, 1], 0, dst_h - 1)
                restored = self.cv2.perspectiveTransform(bu.reshape(1, 4, 2).astype(np.float32), Minv)[0]
                items.append({
                    "polygon": self._expand_polygon(restored, 0.15),
                    "crop_center_x": float(bu[:, 0].mean()),
                    "crop_center_y": float(bu[:, 1].mean()),
                })
            all_items[oi] = items
        return all_items

    # ---------- Text alignment ----------
    def _split_visible_chars(self, text):
        return [ch for ch in text if not ch.isspace()]

    def _attach_text(self, text, char_items):
        if not char_items:
            return []
        items = sorted(char_items, key=lambda x: (x["crop_center_x"], x["crop_center_y"]))
        chars = self._split_visible_chars(text)
        nb, nc = len(items), len(chars)
        if nc == 0:
            for it in items: it["text"] = ""
            return items
        if nb == nc:
            for i, it in enumerate(items): it["text"] = chars[i]
            return items
        if nb < nc:
            cuts = np.floor(np.linspace(0, nc, nb + 1)).astype(int)
            cuts[-1] = nc
            for i, it in enumerate(items):
                s, e = int(cuts[i]), int(cuts[i + 1])
                if e <= s: e = min(s + 1, nc)
                it["text"] = "".join(chars[s:e])
            return items
        for i, it in enumerate(items):
            idx = int(np.clip(np.floor((i + 0.5) * nc / nb), 0, nc - 1))
            it["text"] = chars[idx]
        return items

    # ---------- Process one frame ----------
    def process_frame(self, video_name: str, img_path: str) -> dict:
        frame_name = os.path.splitext(os.path.basename(img_path))[0]
        frame_num = int(frame_name.split("_")[1]) if "_" in frame_name else 0

        image_bgr = self.cv2.imread(img_path)
        if image_bgr is None:
            return {
                "file_name": video_name,
                "frame_num": frame_num,
                "ocr_texts": "[]", "ocr_scores": "[]", "ocr_xywha": "[]",
                "craft_texts": "[]", "craft_xywha": "[]", "craft_parent_idx": "[]",
            }

        result = list(self.ocr.predict(image_bgr))

        ocr_texts, ocr_scores, ocr_xywha = [], [], []
        craft_texts, craft_xywha, craft_parent_idx = [], [], []

        if result:
            page = self._normalize_page_result(result[0])
            dt_polys = page.get("dt_polys", [])
            rec_texts = page.get("rec_texts", [])
            rec_scores = page.get("rec_scores", [])

            polygons = []
            for box, text, score in zip(dt_polys, rec_texts, rec_scores):
                poly = np.array(box, dtype=np.float32)
                ocr_texts.append(text)
                ocr_scores.append(round(float(score), 4))
                ocr_xywha.append(polygon_to_xywha(self.cv2, poly))
                polygons.append(poly)

            if polygons:
                try:
                    all_char = self._segment_polygons_batch(image_bgr, polygons)
                except Exception:
                    all_char = [[] for _ in polygons]

                for li, (text, citems) in enumerate(zip(rec_texts, all_char)):
                    citems = self._attach_text(text, citems)
                    for it in citems:
                        craft_texts.append(it["text"])
                        craft_xywha.append(polygon_to_xywha(self.cv2, it["polygon"]))
                        craft_parent_idx.append(li)

        return {
            "file_name": video_name,
            "frame_num": frame_num,
            "ocr_texts": json.dumps(ocr_texts, ensure_ascii=False),
            "ocr_scores": json.dumps(ocr_scores),
            "ocr_xywha": json.dumps(ocr_xywha),
            "craft_texts": json.dumps(craft_texts, ensure_ascii=False),
            "craft_xywha": json.dumps(craft_xywha),
            "craft_parent_idx": json.dumps(craft_parent_idx),
        }


# -------------------------
# Main
# -------------------------
def main():
    channels = sorted([
        d for d in os.listdir(INPUT_DIR)
        if os.path.isdir(os.path.join(INPUT_DIR, d))
    ])

    print(f"📁 총 {len(channels)}개 채널 발견")

    # 워커 생성
    print(f"🔧 {NUM_WORKERS}개 OCR 워커 생성 중...")
    workers = [OCRWorker.remote() for _ in range(NUM_WORKERS)]

    # 워밍업
    print("🔥 워밍업 중...")
    first_channel_dir = os.path.join(INPUT_DIR, channels[0])
    first_frames = []
    for vn in os.listdir(first_channel_dir):
        vd = os.path.join(first_channel_dir, vn)
        if not os.path.isdir(vd):
            continue
        for f in os.listdir(vd):
            if f.startswith("frame_") and f.endswith(".jpg"):
                first_frames.append((vn, os.path.join(vd, f)))
                break
        if first_frames:
            break
    if first_frames:
        ray.get(workers[0].process_frame.remote(first_frames[0][0], first_frames[0][1]))
    print("✅ 준비 완료!\n")

    for i, channel_name in enumerate(channels):
        channel_dir = os.path.join(INPUT_DIR, channel_name)
        out_dir = os.path.join(OUTPUT_DIR, channel_name)
        os.makedirs(out_dir, exist_ok=True)

        # 영상 목록
        video_names = sorted([
            d for d in os.listdir(channel_dir)
            if os.path.isdir(os.path.join(channel_dir, d))
        ])

        # 이미 완료된 영상 스킵
        done_videos = {
            f[:-8] for f in os.listdir(out_dir)
            if f.endswith(".parquet")
        }
        remaining = [v for v in video_names if v not in done_videos]

        if not remaining:
            print(f"[{i+1}/{len(channels)}] {channel_name}: 스킵 (전부 완료, {len(video_names)}개)")
            continue

        # 남은 영상의 프레임 수집 + 영상별 프레임 수 기록
        all_frames = []
        video_frame_counts = {}  # video_name → expected frame count
        for video_name in remaining:
            video_dir = os.path.join(channel_dir, video_name)
            frame_files = sorted([
                f for f in os.listdir(video_dir)
                if f.startswith("frame_") and f.endswith(".jpg")
            ])
            if not frame_files:
                continue
            video_frame_counts[video_name] = len(frame_files)
            for frame_file in frame_files:
                img_path = os.path.join(video_dir, frame_file)
                all_frames.append((video_name, img_path))

        if not all_frames:
            print(f"[{i+1}/{len(channels)}] {channel_name}: 스킵 (프레임 없음)")
            continue

        # 태스크 분배
        tasks = []
        for j, (video_name, img_path) in enumerate(all_frames):
            worker = workers[j % NUM_WORKERS]
            task = worker.process_frame.remote(video_name, img_path)
            tasks.append(task)

        # 결과 수집 + 영상 완료 시 즉시 저장
        video_results = {}  # video_name → list of row dicts
        n_saved = 0
        total_frames_saved = 0
        desc = f"[{i+1}/{len(channels)}] {channel_name[:25]}"
        with tqdm(total=len(tasks), desc=desc, leave=False) as pbar:
            while tasks:
                ready, tasks = ray.wait(tasks, num_returns=min(32, len(tasks)), timeout=1.0)
                if ready:
                    batch_results = ray.get(ready)
                    pbar.update(len(batch_results))

                    for row in batch_results:
                        vn = row["file_name"]
                        if vn not in video_results:
                            video_results[vn] = []
                        video_results[vn].append(row)

                    # 프레임이 다 모인 영상은 즉시 저장
                    for vn in list(video_results.keys()):
                        if len(video_results[vn]) >= video_frame_counts.get(vn, float("inf")):
                            vdf = pd.DataFrame(video_results.pop(vn))
                            vdf = vdf.drop(columns=["file_name"]).sort_values("frame_num").reset_index(drop=True)
                            out_path = os.path.join(out_dir, f"{vn}.parquet")
                            vdf.to_parquet(out_path, index=False)
                            n_saved += 1
                            total_frames_saved += len(vdf)

        # 혹시 남은 것도 저장 (프레임 수 불일치 등)
        #for vn, rows in video_results.items():
        #    vdf = pd.DataFrame(rows)
        #    vdf = vdf.drop(columns=["file_name"]).sort_values("frame_num").reset_index(drop=True)
        #    out_path = os.path.join(out_dir, f"{vn}.parquet")
        #    vdf.to_parquet(out_path, index=False)
        #    n_saved += 1
        #    total_frames_saved += len(vdf)
        
        print(f"[{i+1}/{len(channels)}] {channel_name}: {total_frames_saved:,}개 프레임, {n_saved}개 영상 ✅")

    ray.shutdown()
    print("\n🎉 전체 완료!")


if __name__ == "__main__":
    main()

2026-03-22 11:51:26,013	INFO worker.py:1786 -- Started a local Ray instance.


📁 총 664개 채널 발견
🔧 8개 OCR 워커 생성 중...
🔥 워밍업 중...


(OCRWorker pid=3339396) Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.
(OCRWorker pid=3339396) /tmp/ipykernel_3330926/3559285261.py:73: UserWarning: `lang` and `ocr_version` will be ignored when model names or model directories are not `None`.
(OCRWorker pid=3339396) Creating model: ('PP-OCRv5_mobile_det', None)
(OCRWorker pid=3339396) Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/taeyoung/.paddlex/official_models/PP-OCRv5_mobile_det`.
(OCRWorker pid=3339396) /home/taeyoung/miniconda3/envs/sub/lib/python3.10/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
(OCRWorker pid=3339396)   warnings.warn(warning_message)
(OCRWorker pid=333

✅ 준비 완료!

[1/664]고기남자: 스킵 (전부 완료, 100개)
[2/664] &TEAM: 스킵 (전부 완료, 100개)
[3/664] 14F 일사에프: 스킵 (전부 완료, 100개)
[4/664] 1MILLION Dance Studio: 스킵 (전부 완료, 100개)
[5/664] 1theK Originals - 원더케이 오리지널: 스킵 (전부 완료, 100개)
[6/664] 1분미만: 스킵 (전부 완료, 100개)
[7/664] 1분요리 뚝딱이형: 스킵 (전부 완료, 100개)
[8/664] 2PM: 스킵 (전부 완료, 100개)
[9/664] 5 분 Tricks: 스킵 (전부 완료, 100개)
[10/664] 815머니톡: 스킵 (전부 완료, 100개)
[11/664] 83부부: 스킵 (전부 완료, 100개)
[12/664] A2O Channel: 스킵 (전부 완료, 100개)
[13/664] AFC Asian Cup: 스킵 (전부 완료, 100개)
[14/664] AKMU: 스킵 (전부 완료, 100개)
[15/664] ALL THE K-POP: 스킵 (전부 완료, 100개)
[16/664] ALLDAY PROJECT: 스킵 (전부 완료, 100개)
[17/664] ARIKITCHEN (아리키친): 스킵 (전부 완료, 100개)
[18/664] ARIRANG K-POP: 스킵 (전부 완료, 100개)
[19/664] ARTBEAT: 스킵 (전부 완료, 100개)
[20/664] ASEAN United FC: 스킵 (전부 완료, 100개)
[21/664] ASTRO 아스트로: 스킵 (전부 완료, 100개)
[22/664] ATEEZ: 스킵 (전부 완료, 100개)
[23/664] Again 가요톱10 _ KBS KPOP Classic: 스킵 (전부 완료, 100개)
[24/664] Allblanc TV: 스킵 (전부 완료, 100개)
[25/664] Apink (에이핑크): 스킵 (전부 완료, 100개)
[26/664] B Man 삐맨: 스킵 (전

[528/664] 오분순삭: 27,651개 프레임, 55개 영상 ✅
(raylet) Warning: More than 5000 tasks are pending submission to actor 8f99b911ad1563cb5a05277401000000. To reduce memory usage, wait for these tasks to finish before sending more.


...

[661/664] 휴먼스토리: 36,420개 프레임, 100개 영상 ✅


[662/664] 흥삼이네 Heungsam's Family: 58,182개 프레임, 100개 영상 ✅


[663/664] 히밥heebab: 59,351개 프레임, 100개 영상 ✅


(OCRWorker pid=3339383) /tmp/ipykernel_3330926/3559285261.py:73: UserWarning: `lang` and `ocr_version` will be ignored when model names or model directories are not `None`. [repeated 7x across cluster]
(OCRWorker pid=3339387) Creating model: ('korean_PP-OCRv5_mobile_rec', None) [repeated 15x across cluster]
(OCRWorker pid=3339387) Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/taeyoung/.paddlex/official_models/korean_PP-OCRv5_mobile_rec`. [repeated 15x across cluster]
(OCRWorker pid=3339383) /home/taeyoung/miniconda3/envs/sub/lib/python3.10/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md [repeated 7x across cluster]
(OCRWorker pid=3339383)   warnings.warn(warning_message) [repeated 7x across cluster]


[664/664] 힙으뜸: 29,260개 프레임, 100개 영상 ✅

🎉 전체 완료!


In [1]:
import os
import pandas as pd
from collections import defaultdict

INPUT_DIR = "./data/2_frame_files"
OUTPUT_DIR = "./data/3_ocr_results"

issues = []
stats = {"channels": 0, "videos_total": 0, "videos_done": 0, "videos_missing": 0, "videos_mismatch": 0}

channels = sorted([
    d for d in os.listdir(INPUT_DIR)
    if os.path.isdir(os.path.join(INPUT_DIR, d))
])

print(f"{'채널':<30s}  {'영상수':>6s}  {'완료':>6s}  {'미완':>6s}  {'불일치':>6s}")
print("-" * 80)

for channel_name in channels:
    channel_dir = os.path.join(INPUT_DIR, channel_name)
    out_dir = os.path.join(OUTPUT_DIR, channel_name)

    video_names = sorted([
        d for d in os.listdir(channel_dir)
        if os.path.isdir(os.path.join(channel_dir, d))
    ])

    ch_done = 0
    ch_missing = 0
    ch_mismatch = 0

    for video_name in video_names:
        video_dir = os.path.join(channel_dir, video_name)
        frame_files = [
            f for f in os.listdir(video_dir)
            if f.startswith("frame_") and f.endswith(".jpg")
        ]
        n_frames = len(frame_files)

        parquet_path = os.path.join(out_dir, f"{video_name}.parquet")
        if not os.path.exists(parquet_path):
            ch_missing += 1
            if n_frames > 0:
                issues.append(("미완료", channel_name, video_name, n_frames, 0))
            continue

        df = pd.read_parquet(parquet_path)
        n_rows = len(df)
        ch_done += 1

        if n_frames != n_rows:
            ch_mismatch += 1
            issues.append(("프레임수 불일치", channel_name, video_name, n_frames, n_rows))
            continue

        # frame_num 연속성 체크
        expected_nums = sorted([
            int(f.split("_")[1].split(".")[0]) for f in frame_files
        ])
        actual_nums = sorted(df["frame_num"].tolist())
        if expected_nums != actual_nums:
            ch_mismatch += 1
            missing = set(expected_nums) - set(actual_nums)
            extra = set(actual_nums) - set(expected_nums)
            detail = ""
            if missing:
                detail += f" 누락={sorted(missing)[:5]}"
            if extra:
                detail += f" 초과={sorted(extra)[:5]}"
            issues.append(("frame_num 불일치", channel_name, video_name, n_frames, n_rows, detail))

    print(f"{channel_name:<30s}  {len(video_names):>6d}  {ch_done:>6d}  {ch_missing:>6d}  {ch_mismatch:>6d}")
    stats["channels"] += 1
    stats["videos_total"] += len(video_names)
    stats["videos_done"] += ch_done
    stats["videos_missing"] += ch_missing
    stats["videos_mismatch"] += ch_mismatch

print("-" * 80)
print(f"{'합계':<30s}  {stats['videos_total']:>6d}  {stats['videos_done']:>6d}  {stats['videos_missing']:>6d}  {stats['videos_mismatch']:>6d}")
print()

if issues:
    print(f"⚠️  이상 항목 {len(issues)}건:")
    print(f"  {'유형':<20s}  {'채널':<25s}  {'영상':<30s}  {'프레임':>6s}  {'parquet':>7s}  {'비고'}")
    print("  " + "-" * 110)
    for item in issues:
        typ = item[0]
        ch = item[1]
        vn = item[2]
        nf = item[3]
        nr = item[4]
        note = item[5] if len(item) > 5 else ""
        print(f"  {typ:<20s}  {ch:<25s}  {vn:<30s}  {nf:>6d}  {nr:>7d}  {note}")
else:
    print("✅ 이상 없음. 모든 parquet이 프레임 수와 일치합니다.")

채널                                 영상수      완료      미완     불일치
--------------------------------------------------------------------------------
고기남자                              100     100       0       0
&TEAM                              100     100       0       0
14F 일사에프                           100     100       0       0
1MILLION Dance Studio              100     100       0       0
1theK Originals - 원더케이 오리지널        100     100       0       0
1분미만                               100     100       0       0
1분요리 뚝딱이형                          100     100       0       0
2PM                                100     100       0       0
5 분 Tricks                         100     100       0       0
815머니톡                             100     100       0       0
83부부                               100     100       0       0
A2O Channel                        100     100       0       0
AFC Asian Cup                      100     100       0       0
AKMU                               10

In [ ]:
import os
print(os.environ.get("LD_LIBRARY_PATH"))

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path

OUTPUT_DIR = "./data/3_ocr_results"
CHANNEL = "14F 일사에프"

ch_dir = os.path.join(OUTPUT_DIR, CHANNEL)
parquets = sorted([f for f in os.listdir(ch_dir) if f.endswith(".parquet")])
print(f"📂 {CHANNEL}: {len(parquets)} parquet files\n")

# ---- 1. 기본 통계 ----
total_frames = 0
total_ocr_lines = 0
total_craft_chars = 0
empty_frames = 0
all_scores = []
errors = []

for pf in parquets:
    path = os.path.join(ch_dir, pf)
    try:
        df = pd.read_parquet(path)
    except Exception as e:
        errors.append((pf, f"Read error: {e}"))
        continue

    # 컬럼 체크
    expected_cols = {"frame_num", "ocr_texts", "ocr_scores", "ocr_xywha",
                     "craft_texts", "craft_xywha", "craft_parent_idx"}
    if not expected_cols.issubset(set(df.columns)):
        errors.append((pf, f"Missing columns: {expected_cols - set(df.columns)}"))
        continue

    total_frames += len(df)

    for _, row in df.iterrows():
        try:
            texts = json.loads(row["ocr_texts"])
            scores = json.loads(row["ocr_scores"])
            xywha = json.loads(row["ocr_xywha"])
            c_texts = json.loads(row["craft_texts"])
            c_xywha = json.loads(row["craft_xywha"])
            c_pidx = json.loads(row["craft_parent_idx"])
        except json.JSONDecodeError as e:
            errors.append((pf, f"JSON decode error frame {row['frame_num']}: {e}"))
            continue

        n_lines = len(texts)
        n_chars = len(c_texts)
        total_ocr_lines += n_lines
        total_craft_chars += n_chars

        if n_lines == 0:
            empty_frames += 1

        all_scores.extend(scores)

        # 일관성 체크
        if len(scores) != n_lines:
            errors.append((pf, f"frame {row['frame_num']}: scores({len(scores)}) != texts({n_lines})"))
        if len(xywha) != n_lines:
            errors.append((pf, f"frame {row['frame_num']}: xywha({len(xywha)}) != texts({n_lines})"))
        if len(c_xywha) != n_chars:
            errors.append((pf, f"frame {row['frame_num']}: craft_xywha({len(c_xywha)}) != craft_texts({n_chars})"))
        if len(c_pidx) != n_chars:
            errors.append((pf, f"frame {row['frame_num']}: craft_parent_idx({len(c_pidx)}) != craft_texts({n_chars})"))

        # parent_idx 범위 체크
        if c_pidx and n_lines > 0:
            if max(c_pidx) >= n_lines:
                errors.append((pf, f"frame {row['frame_num']}: parent_idx max({max(c_pidx)}) >= n_lines({n_lines})"))

print("=" * 60)
print("📊 기본 통계")
print("=" * 60)
print(f"   Parquet files:    {len(parquets)}")
print(f"   Total frames:     {total_frames:,}")
print(f"   Empty frames:     {empty_frames:,} ({100*empty_frames/max(total_frames,1):.1f}%)")
print(f"   OCR lines:        {total_ocr_lines:,} (avg {total_ocr_lines/max(total_frames,1):.1f}/frame)")
print(f"   CRAFT chars:      {total_craft_chars:,} (avg {total_craft_chars/max(total_frames,1):.1f}/frame)")

if all_scores:
    scores_arr = np.array(all_scores)
    print(f"\n📈 OCR Score 분포")
    print(f"   mean: {scores_arr.mean():.3f}")
    print(f"   min:  {scores_arr.min():.3f}")
    print(f"   25%:  {np.percentile(scores_arr, 25):.3f}")
    print(f"   50%:  {np.percentile(scores_arr, 50):.3f}")
    print(f"   75%:  {np.percentile(scores_arr, 75):.3f}")
    print(f"   max:  {scores_arr.max():.3f}")
    print(f"   < 0.5: {(scores_arr < 0.5).sum():,} ({100*(scores_arr < 0.5).mean():.1f}%)")

print(f"\n🔍 에러/경고: {len(errors)}개")
if errors:
    for pf, msg in errors[:20]:
        print(f"   [{pf[:40]}] {msg}")
    if len(errors) > 20:
        print(f"   ... +{len(errors) - 20}개 더")
else:
    print("   없음 ✅")

# ---- 2. 샘플 확인 ----
print(f"\n{'=' * 60}")
print("🔎 샘플 확인 (첫 번째 파일, 첫 5프레임)")
print("=" * 60)
sample_path = os.path.join(ch_dir, parquets[0])
sdf = pd.read_parquet(sample_path)
print(f"File: {parquets[0]}")
print(f"Shape: {sdf.shape}\n")

for _, row in sdf.head(5).iterrows():
    texts = json.loads(row["ocr_texts"])
    scores = json.loads(row["ocr_scores"])
    c_texts = json.loads(row["craft_texts"])
    c_pidx = json.loads(row["craft_parent_idx"])
    print(f"  Frame {row['frame_num']:>5}: {len(texts)} lines, {len(c_texts)} chars")
    for i, (t, s) in enumerate(zip(texts, scores)):
        # 해당 라인의 craft 문자들
        chars = [c_texts[j] for j in range(len(c_pidx)) if c_pidx[j] == i]
        print(f"    [{s:.2f}] {t}  →  {''.join(chars)}")

In [3]:
import os
import json
import pandas as pd
from glob import glob
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

DATA_DIR = "./data/3_ocr_results"
NUM_WORKERS = 32


def process_channel(ch):
    ch_dir = os.path.join(DATA_DIR, ch)
    parquets = [os.path.join(ch_dir, f) for f in os.listdir(ch_dir) if f.endswith(".parquet")]
    ch_frames = 0
    ch_bboxes = 0
    for pq in parquets:
        df = pd.read_parquet(pq)
        ch_frames += len(df)
        ch_bboxes += sum(len(json.loads(t)) for t in df["ocr_texts"])
    return {"channel": ch, "videos": len(parquets), "frames": ch_frames, "ocr_bboxes": ch_bboxes}


channels = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])

rows = []
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = {executor.submit(process_channel, ch): ch for ch in channels}
    for future in tqdm(as_completed(futures), total=len(channels), desc="채널"):
        rows.append(future.result())

rows.sort(key=lambda x: x["channel"])

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print(f"\n총 채널: {len(channels)}")
print(f"총 영상: {summary['videos'].sum():,}")
print(f"총 프레임: {summary['frames'].sum():,}")
print(f"총 OCR bbox: {summary['ocr_bboxes'].sum():,}")

채널: 100%|██████████| 664/664 [00:13<00:00, 50.69it/s]


                                channel  videos  frames  ocr_bboxes
                                 고기남자     100   55879       97534
                                  &TEAM     100   21549       31279
                               14F 일사에프     100   52470      321576
                  1MILLION Dance Studio     100   37486      114098
            1theK Originals - 원더케이 오리지널     100   31621      157335
                                   1분미만     100   43592      444257
                              1분요리 뚝딱이형     100   58545      111564
                                    2PM     100   34727      121850
                             5 분 Tricks     100   38172       61463
                                 815머니톡     100   41316      359672
                                   83부부     100   51804      105558
                            A2O Channel     100   22598       49790
                          AFC Asian Cup     100   24208      100937
                                   AKMU     100   